# 01 Tokenization / Vocabulary / Padding / Mask

目标：亲手走一遍

```text
Text → Token → Vocabulary → ID → Padding → Mask
```

本 Notebook 不依赖真实 IMDB 数据。


In [ ]:
import re
import torch
from collections import Counter

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

## 1. Tokenization

In [ ]:
def tokenize(text: str):
    text = text.lower()
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\w\s]", text)

sentences = [
    "I love deep learning.",
    "I really love PyTorch!",
    "Deep learning loves data.",
]

tokenized = [tokenize(sentence) for sentence in sentences]
tokenized

## 2. 构建 Vocabulary

In [ ]:
counter = Counter(token for sentence in tokenized for token in sentence)

idx_to_token = [PAD_TOKEN, UNK_TOKEN] + [
    token for token, _ in counter.most_common()
]

token_to_idx = {
    token: index
    for index, token in enumerate(idx_to_token)
}

token_to_idx

## 3. Token → ID

In [ ]:
def encode(tokens):
    return [
        token_to_idx.get(token, UNK_ID)
        for token in tokens
    ]

encoded = [encode(tokens) for tokens in tokenized]
encoded

## 4. Padding

In [ ]:
max_length = max(len(ids) for ids in encoded)

batch = torch.full(
    (len(encoded), max_length),
    fill_value=PAD_ID,
    dtype=torch.long,
)

lengths = []

for row, ids in enumerate(encoded):
    ids_tensor = torch.tensor(ids, dtype=torch.long)
    batch[row, : len(ids)] = ids_tensor
    lengths.append(len(ids))

lengths = torch.tensor(lengths)

print("batch shape:", batch.shape)
print(batch)
print("lengths:", lengths)

## 5. Mask

In [ ]:
mask = batch.ne(PAD_ID)

print(mask)
print("mask shape:", mask.shape)
print("每条真实 token 数：", mask.sum(dim=1))

## 复习

- `batch.shape = [B, L]`
- Padding 只是为了凑成矩阵；
- `lengths` 常给 RNN；
- `mask` 常给 pooling / attention；
- `<unk>` 处理词表外 token。
